# Silver Layer — Account Processing

## Objective
Prepare clean and consistent banking account data for downstream analytics.

## Implementation
- Process account changes incrementally from Bronze using Delta Change Data Feed (CDF).
- Apply data cleaning and deduplication based on account ID.
- Use Delta MERGE to maintain new and updated account records.
- Provide account data for the Gold dimension table.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

bronze_table = "banking_project.bronze.bronze_accounts"
silver_table = "banking_project.silver.silver_accounts"

checkpoint_path = (
    "/Volumes/banking_project/silver/"
    "silver_pipeline_state/accounts_cdf_checkpoint"
)

def merge_accounts(batch_df, batch_id):

    if batch_df.isEmpty():
        return

    # Read inserted and updated records
    df = batch_df.filter(
        F.col("_change_type").isin("insert", "update_postimage")
    )

    # Clean and validate
    df = (
        df.select(
            F.trim("account_id").alias("account_id"),
            F.trim("customer_id").alias("customer_id"),
            F.trim("account_type").alias("account_type"),
            "ingestion_timestamp",
            "source_file",
            "_commit_version"
        )
        .filter(
            F.col("account_id").isNotNull() &
            (F.col("account_id") != "") &
            F.col("customer_id").isNotNull() &
            (F.col("customer_id") != "")
        )
    )

    # Keep latest record for each account
    window = Window.partitionBy("account_id").orderBy(
        F.col("_commit_version").desc(),
        F.col("ingestion_timestamp").desc_nulls_last()
    )

    df = (
        df.withColumn("rn", F.row_number().over(window))
        .filter(F.col("rn") == 1)
        .drop("rn", "_commit_version")
    )

    # MERGE into Silver Accounts
    (
        DeltaTable.forName(spark, silver_table)
        .alias("target")
        .merge(
            df.alias("source"),
            "target.account_id = source.account_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

query = (
    spark.readStream
    .option("readChangeFeed", "true")
    .table(bronze_table)
    .writeStream
    .foreachBatch(merge_accounts)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

print("Silver Accounts CDF processing completed")